# InsureGig — Disruption Prediction Model

**Goal:** Train a multi-output neural network that predicts disruption severity across 4 categories:
- `weather` — extreme rainfall / storm events
- `aqi` — hazardous air quality
- `traffic` — congestion gridlock
- `platform` — delivery app outage probability

**Input:** 16 engineered features (temporal cyclical + city location + AQI lags + rainfall lags)

**Output:** 4 severity scores, each in [0, 1]

**Export:** TF.js LayersModel → `public/disruption_model/` in your React app

---

### Datasets used
| Dataset | Source | What it provides |
|---------|--------|------------------|
| India AQI 2010–2023 | Kaggle | City-level daily PM2.5, PM10, AQI |
| IMD Rainfall India | Kaggle | Sub-division monthly rainfall (mm) |
| EM-DAT India disasters | emdat.be | Historical flood/cyclone event dates |
| Uber Movement Bangalore | movement.uber.com | Hourly zone-to-zone travel ratios |

> **Platform outages** have no public CSV — we generate synthetic labels using a Poisson process calibrated to ~3 major outages/month industry-wide.


In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
!pip install -q kaggle tensorflowjs pandas scikit-learn numpy matplotlib seaborn

In [ ]:
# ── 2. Kaggle API setup ────────────────────────────────────────────────────────
# Upload your kaggle.json here (from https://www.kaggle.com/settings → API → Create Token)
from google.colab import files
import os

print('Upload your kaggle.json...')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle API ready.')

In [ ]:
# ── 3. Download datasets ───────────────────────────────────────────────────────
!kaggle datasets download -d abhisheksjha/time-series-air-quality-data-of-india-2010-2023 -p /data/aqi --unzip
!kaggle datasets download -d rajanand/rainfall-in-india -p /data/rainfall --unzip

# List what we got
import os
for root, dirs, files_ in os.walk('/data'):
    for f in files_:
        print(os.path.join(root, f))

In [ ]:
# ── 4. Load and inspect AQI dataset ───────────────────────────────────────────
import pandas as pd
import numpy as np
import glob

# The AQI dataset may be split by city — load all CSVs
aqi_files = glob.glob('/data/aqi/**/*.csv', recursive=True)
print(f'Found {len(aqi_files)} AQI files')

aqi_dfs = []
for f in aqi_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        aqi_dfs.append(df)
    except Exception as e:
        print(f'Skipping {f}: {e}')

aqi_raw = pd.concat(aqi_dfs, ignore_index=True)
print(f'AQI dataset shape: {aqi_raw.shape}')
print(aqi_raw.columns.tolist())
aqi_raw.head(3)

In [ ]:
# ── 5. Normalize AQI column names ─────────────────────────────────────────────
# Different Kaggle datasets use slightly different column names — normalize them
aqi_raw.columns = aqi_raw.columns.str.lower().str.strip().str.replace(' ', '_')
print(aqi_raw.columns.tolist())

# Find the date column
date_col = next((c for c in aqi_raw.columns if 'date' in c or 'time' in c), None)
city_col = next((c for c in aqi_raw.columns if 'city' in c or 'station' in c or 'location' in c), None)
aqi_col  = next((c for c in aqi_raw.columns if c == 'aqi' or c == 'aqi_index'), None)
pm25_col = next((c for c in aqi_raw.columns if 'pm2' in c or 'pm25' in c), None)

print(f'date={date_col}, city={city_col}, aqi={aqi_col}, pm25={pm25_col}')

In [ ]:
# ── 6. Clean AQI dataset ───────────────────────────────────────────────────────
aqi = aqi_raw[[date_col, city_col, aqi_col, pm25_col]].copy()
aqi.columns = ['date', 'city', 'aqi', 'pm25']
aqi['date'] = pd.to_datetime(aqi['date'], errors='coerce')
aqi['aqi']  = pd.to_numeric(aqi['aqi'],  errors='coerce')
aqi['pm25'] = pd.to_numeric(aqi['pm25'], errors='coerce')
aqi.dropna(subset=['date', 'aqi'], inplace=True)

# Group to daily (some datasets are hourly)
aqi = aqi.groupby(['city', pd.Grouper(key='date', freq='D')]).agg(
    aqi=('aqi', 'mean'),
    pm25=('pm25', 'mean')
).reset_index()

# Focus on major delivery cities
TARGET_CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Kolkata', 'Pune', 'Ahmedabad']
aqi['city_match'] = aqi['city'].str.extract(f'({"|".join(TARGET_CITIES)})', expand=False, flags=2)
aqi = aqi[aqi['city_match'].notna()].copy()
aqi['city'] = aqi['city_match']
aqi.drop(columns=['city_match'], inplace=True)

print(f'Cleaned AQI shape: {aqi.shape}')
print(aqi['city'].value_counts())

In [ ]:
# ── 7. Load and clean Rainfall dataset ────────────────────────────────────────
rainfall_files = glob.glob('/data/rainfall/**/*.csv', recursive=True)
print(f'Found {len(rainfall_files)} rainfall files')

rain_raw = pd.read_csv(rainfall_files[0])
print(rain_raw.columns.tolist())
rain_raw.head(3)

In [ ]:
# ── 8. Map rainfall subdivisions → cities ─────────────────────────────────────
# IMD data is by meteorological sub-division; map to our target cities
SUBDIVISION_MAP = {
    'Konkan & Goa':          'Mumbai',
    'Gujarat Region':        'Ahmedabad',
    'Coastal Karnataka':     'Bangalore',
    'Interior Karnataka':    'Bangalore',
    'Tamil Nadu':            'Chennai',
    'Coastal Andhra Pradesh':'Hyderabad',
    'Telangana':             'Hyderabad',
    'West Bengal & Sikkim':  'Kolkata',
    'Sub-Himalayan West Bengal & Sikkim': 'Kolkata',
    'Haryana Delhi & Chandigarh':   'Delhi',
    'West Uttar Pradesh':    'Delhi',
    'Marathwada':            'Pune',
    'Madhya Maharashtra':    'Pune',
}

# Melt monthly columns into rows
MONTHS = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
available_months = [m for m in MONTHS if m in rain_raw.columns]
rain_melted = rain_raw.melt(
    id_vars=['SUBDIVISION', 'YEAR'],
    value_vars=available_months,
    var_name='month_abbr',
    value_name='rainfall_mm'
)
rain_melted['month'] = pd.to_datetime(rain_melted['month_abbr'], format='%b').dt.month
rain_melted['city'] = rain_melted['SUBDIVISION'].map(SUBDIVISION_MAP)
rain_melted.dropna(subset=['city', 'rainfall_mm'], inplace=True)
rain_melted['rainfall_mm'] = pd.to_numeric(rain_melted['rainfall_mm'], errors='coerce')

# Group to city+year+month
rain = rain_melted.groupby(['city', 'YEAR', 'month'])['rainfall_mm'].mean().reset_index()
rain.columns = ['city', 'year', 'month', 'rainfall_mm']
print(f'Rainfall shape: {rain.shape}')
rain.head(3)

In [ ]:
# ── 9. Merge AQI + Rainfall on city + year + month ────────────────────────────
aqi['year']  = aqi['date'].dt.year
aqi['month'] = aqi['date'].dt.month
aqi['day']   = aqi['date'].dt.day

merged = aqi.merge(rain, on=['city', 'year', 'month'], how='left')
merged['rainfall_mm'].fillna(merged.groupby(['city', 'month'])['rainfall_mm'].transform('median'), inplace=True)
merged['rainfall_mm'].fillna(0, inplace=True)

print(f'Merged dataset: {merged.shape}')
merged[['city', 'date', 'aqi', 'pm25', 'rainfall_mm']].head(5)

In [ ]:
# ── 10. City GPS coordinates for location features ────────────────────────────
CITY_COORDS = {
    'Mumbai':    (19.0760,  72.8777),
    'Delhi':     (28.6139,  77.2090),
    'Bangalore': (12.9716,  77.5946),
    'Hyderabad': (17.3850,  78.4867),
    'Chennai':   (13.0827,  80.2707),
    'Kolkata':   (22.5726,  88.3639),
    'Pune':      (18.5204,  73.8567),
    'Ahmedabad': (23.0225,  72.5714),
}
merged['lat'] = merged['city'].map(lambda c: CITY_COORDS.get(c, (20.0, 78.0))[0])
merged['lon'] = merged['city'].map(lambda c: CITY_COORDS.get(c, (20.0, 78.0))[1])

# Normalize lat/lon to [0, 1]
merged['lat_norm'] = (merged['lat'] - 8.0)  / (37.0 - 8.0)
merged['lon_norm'] = (merged['lon'] - 68.0) / (97.5 - 68.0)

In [ ]:
# ── 11. Feature engineering ────────────────────────────────────────────────────
df = merged.sort_values(['city', 'date']).copy()

# Cyclical temporal encodings
df['hour'] = 12  # monthly data — assume midday
df['hour_sin'] = np.sin(2 * np.pi * df['hour']   / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']   / 24)
df['month_sin']= np.sin(2 * np.pi * df['month']  / 12)
df['month_cos']= np.cos(2 * np.pi * df['month']  / 12)
df['dow']      = df['date'].dt.dayofweek
df['dow_sin']  = np.sin(2 * np.pi * df['dow']    / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['dow']    / 7)
df['is_monsoon']   = df['month'].isin([6, 7, 8, 9]).astype(float)
df['is_winter_fog']= df['month'].isin([11, 12, 1, 2]).astype(float)

# Lag features per city
for lag in [1, 3, 7]:
    df[f'aqi_lag{lag}']      = df.groupby('city')['aqi'].shift(lag)
    df[f'rain_lag{lag}']     = df.groupby('city')['rainfall_mm'].shift(lag)

# Rolling means
df['aqi_roll7']  = df.groupby('city')['aqi'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df['rain_roll7'] = df.groupby('city')['rainfall_mm'].transform(lambda x: x.rolling(7, min_periods=1).mean())

# Normalize continuous features
df['aqi_norm']      = np.clip(df['aqi']         / 500.0, 0, 1)
df['pm25_norm']     = np.clip(df['pm25']        / 300.0, 0, 1)
df['rain_norm']     = np.clip(df['rainfall_mm'] / 200.0, 0, 1)
df['aqi_lag1_norm'] = np.clip(df['aqi_lag1']    / 500.0, 0, 1)
df['aqi_lag7_norm'] = np.clip(df['aqi_lag7']    / 500.0, 0, 1)
df['rain_lag1_norm']= np.clip(df['rain_lag1']   / 200.0, 0, 1)
df['rain_lag7_norm']= np.clip(df['rain_lag7']   / 200.0, 0, 1)
df['aqi_roll7_norm']= np.clip(df['aqi_roll7']   / 500.0, 0, 1)
df['rain_roll7_norm']= np.clip(df['rain_roll7'] / 200.0, 0, 1)

df.dropna(subset=['aqi_lag1_norm', 'rain_lag1_norm'], inplace=True)
print(f'Feature-engineered dataset: {df.shape}')

In [ ]:
# ── 12. Create disruption severity labels ──────────────────────────────────────
# weather_severity: driven by rainfall (heavy rain = high severity)
df['label_weather'] = np.clip(
    (df['rainfall_mm'] - 10) / 150.0, 0, 1
).fillna(0)

# aqi_severity: driven by AQI (WHO unsafe = AQI > 100)
df['label_aqi'] = np.clip(
    (df['aqi'] - 100) / 300.0, 0, 1
).fillna(0)

# traffic_severity: proxy — higher during monsoon + business hours + Metro cities
# Delhi/Mumbai/Bangalore have higher baseline traffic
METRO_TRAFFIC = {'Delhi': 0.75, 'Mumbai': 0.80, 'Bangalore': 0.70,
                 'Hyderabad': 0.55, 'Chennai': 0.60, 'Kolkata': 0.65,
                 'Pune': 0.50, 'Ahmedabad': 0.45}
df['city_traffic_base'] = df['city'].map(METRO_TRAFFIC).fillna(0.5)
# Traffic worsens with rain and peaks mid-week
df['label_traffic'] = np.clip(
    df['city_traffic_base'] * (1 + df['rain_norm'] * 0.4) * (1 - df['is_monsoon'] * 0.1),
    0, 1
)

# platform_outage: rare events — Poisson synthetic labels
# ~3 outages/month industry-wide, ~20% severity on average
np.random.seed(42)
n = len(df)
outage_events = np.random.poisson(0.1, n)  # ~10% days have partial outage
outage_severity = np.random.beta(1.5, 5, n)  # skewed low — most are minor
df['label_platform'] = np.where(outage_events > 0, outage_severity, 0.0)

# Add known major outage dates (Zomato+Swiggy AWS failure Apr 2022)
KNOWN_OUTAGES = ['2022-04-06', '2021-08-17', '2020-11-25', '2023-03-15']
for od in KNOWN_OUTAGES:
    mask = df['date'] == pd.Timestamp(od)
    df.loc[mask, 'label_platform'] = 0.85

print('Label distributions:')
for label in ['label_weather', 'label_aqi', 'label_traffic', 'label_platform']:
    print(f'  {label}: mean={df[label].mean():.3f}, std={df[label].std():.3f}')

In [ ]:
# ── 13. Assemble feature matrix ────────────────────────────────────────────────
FEATURES = [
    # Temporal (cyclical)
    'month_sin', 'month_cos',
    'dow_sin',   'dow_cos',
    # Seasons
    'is_monsoon', 'is_winter_fog',
    # Location
    'lat_norm', 'lon_norm',
    # AQI features
    'aqi_norm', 'aqi_lag1_norm', 'aqi_lag7_norm', 'aqi_roll7_norm',
    # Rainfall features
    'rain_norm', 'rain_lag1_norm', 'rain_lag7_norm', 'rain_roll7_norm',
]

LABELS = ['label_weather', 'label_aqi', 'label_traffic', 'label_platform']

X = df[FEATURES].values.astype(np.float32)
Y = df[LABELS].values.astype(np.float32)

print(f'X shape: {X.shape}  →  {len(FEATURES)} features')
print(f'Y shape: {Y.shape}  →  4 disruption severity outputs')
print(f'\nFeature list: {FEATURES}')

In [ ]:
# ── 14. Train / val split ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.15, random_state=42, shuffle=True
)
print(f'Train: {X_train.shape[0]} samples | Val: {X_val.shape[0]} samples')

In [ ]:
# ── 15. Build the model ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

tf.random.set_seed(42)

def build_disruption_model(n_features: int = 16) -> keras.Model:
    inp = keras.Input(shape=(n_features,), name='features')

    # Shared backbone
    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Dense(16, activation='relu')(x)

    # Separate output head per disruption type
    # This lets each type learn its own feature weighting
    weather  = layers.Dense(8, activation='relu')(x)
    weather  = layers.Dense(1, activation='sigmoid', name='weather')(weather)

    aqi      = layers.Dense(8, activation='relu')(x)
    aqi      = layers.Dense(1, activation='sigmoid', name='aqi')(aqi)

    traffic  = layers.Dense(8, activation='relu')(x)
    traffic  = layers.Dense(1, activation='sigmoid', name='traffic')(traffic)

    platform = layers.Dense(8, activation='relu')(x)
    platform = layers.Dense(1, activation='sigmoid', name='platform')(platform)

    out = layers.Concatenate(name='disruption_scores')([weather, aqi, traffic, platform])

    model = keras.Model(inputs=inp, outputs=out, name='InsureGig_DisruptionModel')
    return model

model = build_disruption_model(n_features=len(FEATURES))
model.summary()

In [ ]:
# ── 16. Compile and train ──────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mean_squared_error',
    metrics=['mae']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5
    ),
    keras.callbacks.ModelCheckpoint(
        '/disruption_model_best.keras', save_best_only=True
    ),
]

history = model.fit(
    X_train, Y_train,
    epochs=120,
    batch_size=256,
    validation_data=(X_val, Y_val),
    callbacks=callbacks,
    verbose=1
)

val_loss = min(history.history['val_loss'])
print(f'\nBest val_loss: {val_loss:.4f}')

In [ ]:
# ── 17. Evaluate per-output ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

Y_pred = model.predict(X_val)
LABEL_NAMES = ['weather', 'aqi', 'traffic', 'platform']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, name) in enumerate(zip(axes, LABEL_NAMES)):
    ax.scatter(Y_val[:, i], Y_pred[:, i], alpha=0.3, s=5)
    ax.plot([0, 1], [0, 1], 'r--', lw=1)
    mae = np.mean(np.abs(Y_val[:, i] - Y_pred[:, i]))
    ax.set_title(f'{name}\nMAE={mae:.3f}')
    ax.set_xlabel('True'); ax.set_ylabel('Predicted')
plt.tight_layout()
plt.savefig('/disruption_eval.png', dpi=100)
plt.show()
print('Evaluation saved.')

In [ ]:
# ── 18. Export to TensorFlow.js format ────────────────────────────────────────
import tensorflowjs as tfjs

TFJS_OUTPUT = '/disruption_model_tfjs'
tfjs.converters.save_keras_model(model, TFJS_OUTPUT)

print('TF.js model files:')
for f in os.listdir(TFJS_OUTPUT):
    size = os.path.getsize(os.path.join(TFJS_OUTPUT, f))
    print(f'  {f}  ({size/1024:.1f} KB)')

In [ ]:
# ── 19. Save feature metadata (required for browser inference) ─────────────────
import json

meta = {
    'features': FEATURES,
    'outputs': LABEL_NAMES,
    'n_features': len(FEATURES),
    'city_coords': CITY_COORDS,
    'normalization': {
        'aqi_max': 500,
        'pm25_max': 300,
        'rainfall_max': 200,
        'lat_range': [8.0, 37.0],
        'lon_range': [68.0, 97.5]
    },
    'thresholds': {
        'weather_trigger': 0.40,
        'aqi_trigger':     0.35,
        'traffic_trigger': 0.55,
        'platform_trigger':0.30
    }
}

with open(f'{TFJS_OUTPUT}/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('model_meta.json saved.')

In [ ]:
# ── 20. Zip and download ───────────────────────────────────────────────────────
import shutil

shutil.make_archive('/disruption_model_tfjs', 'zip', '/disruption_model_tfjs')

from google.colab import files
files.download('/disruption_model_tfjs.zip')
print('\nDownload the zip, then extract contents into:')
print('  your-project/public/disruption_model/')
print('  (should contain model.json, group1-shard*.bin, model_meta.json)')

## After downloading

1. Unzip `disruption_model_tfjs.zip`
2. Copy all files into `public/disruption_model/` in your InsureGig project
3. The app automatically loads it via `loadDisruptionModel()` in `mlEngine.ts`
4. `risk-insights-page.tsx` will now show data-driven forecasts instead of static placeholders

### Re-training
- Run this notebook again after downloading new AQI data (`kaggle datasets download ...`)
- The model trains in ~5 min on Colab T4 GPU with 50k+ samples
- Accuracy improves significantly with 2+ years of daily city data
